### I am mainly using this notebook for function testing/development prior to adding them to their respective module

In [1]:
from dotenv import load_dotenv
import os

import state.kalshi_helpers as kh
import state.coingecko_helpers as ch
import state.influx_helpers as ih

import requests
from datetime import datetime, timezone




In [ ]:

# ---------- KALSHI IDENTIFICATION INFORMATION ----------
KALSHI_ID= os.environ["KALSHI_KEY_ID"]
KEY_PATH = os.environ["KALSHI_PEM_PATH"]
KALSHI_KEY = kh.load_private_key_from_file(KEY_PATH)


#these are here for testing only

BASE_URL = "https://api.elections.kalshi.com"
PUBLIC_MARKETS_URL = f"https://{BASE_URL}/trade-api/v2/markets"
WS_URL = "wss://api.elections.kalshi.com/trade-api/ws/v2"
BITCOIN_TICKER = "KXBTC15M"



# ---------- INFLUX VARIABELS INFORMATION ----------
kalshi_id, kalshi_key = kh.load_kalshi_vars()
username, password, bucket, token, org = ih.load_influx_vars()





In [4]:
from dotenv import load_dotenv
from influxdb_client import InfluxDBClient
from state.influx_helpers import get_influx_client, get_write_api, load_influx_vars
from state.kalshi_helpers import get_market_snapshot
import os

load_dotenv()

# influx setup
_, _, bucket, token, org = load_influx_vars()
client = get_influx_client("http://localhost:8086", token, org)
write_api = get_write_api(client)

# fetch a real snapshot
active_tick = kh.get_active_market(BITCOIN_TICKER, limit=1000)

snapshot = get_market_snapshot(active_tick)  



ih.wrt_kalshi_market_snapshot(write_api, bucket, org, snapshot)
print("written")

ACTIVE MARKET | BTC price up in next 15 mins? | close time: 2026-03-18T18:45:00Z | active
written


In [ ]:

kalshi_id, kalshi_key = kh.load_kalshi_vars()


orderbook = kh.get_market_orderbook(active_tick, kalshi_id, kalshi_key)


_, _, bucket, token, org = load_influx_vars()

client = get_influx_client("http://localhost:8086", token, org)
write_api = get_write_api(client)

ih.wrt_kalshi_orderbook(write_api, bucket, org, active_tick, orderbook=orderbook)





#




ACTIVE MARKET | BTC price up in next 15 mins? | close time: 2026-03-18T18:00:00Z | active


In [ ]:
btc_dict = ch.get_coin_snapshot('bitcoin')
ih.write_coingecko_snapshot(write_api, bucket, org, data=btc_dict)

In [8]:
import asyncio

active_tick = kh.get_active_market(BITCOIN_TICKER)
queue = asyncio.Queue()
await kh.websocket_ingest(active_tick, BITCOIN_TICKER, kalshi_id, kalshi_key)

ACTIVE MARKET | BTC price up in next 15 mins? | close time: 2026-03-18T19:30:00Z | active
[WS] Connected to Kalshi for KXBTC15M-26MAR181530-30
{'ts': datetime.datetime(2026, 3, 18, 19, 19, 21, 28171, tzinfo=datetime.timezone.utc), 'source': 'kalshi_ws', 'market_ticker': 'KXBTC15M-26MAR181530-30', 'series_ticker': ['KXBTC15M'], 'payload': {'type': 'subscribed', 'id': 1, 'msg': {'channel': 'orderbook_delta', 'sid': 1}}}
{'ts': datetime.datetime(2026, 3, 18, 19, 19, 21, 28278, tzinfo=datetime.timezone.utc), 'source': 'kalshi_ws', 'market_ticker': 'KXBTC15M-26MAR181530-30', 'series_ticker': ['KXBTC15M'], 'payload': {'type': 'orderbook_snapshot', 'sid': 1, 'seq': 1, 'msg': {'market_ticker': 'KXBTC15M-26MAR181530-30', 'market_id': '311ee745-c885-430e-8794-1d30d9f7c736', 'yes_dollars_fp': [['0.0010', '5200.00'], ['0.0020', '2500.00'], ['0.0100', '7539.00'], ['0.0200', '3795.00'], ['0.0300', '231.00'], ['0.0400', '655.00'], ['0.0500', '224.00'], ['0.0600', '80039.00'], ['0.0700', '138.00'], ['

CancelledError: 